# 15 · Outlook — unfitted FEM with ngsxfem 🫧

A **very advanced** closing glimpse. We build a two-phase **rising-bubble** solver (à la
**Hysing–Turek**) *from scratch* with **NGSolve + ngsxfem** — introducing every unfitted-FEM
concept step by step, always the *simplest* variant. Two **supplements** then take the
production route with **ngsxditto**: **(A)** a robust **mean-curvature** surface-tension
solver, and **(B)** a **fine-mesh** benchmark run with animation.

**Roadmap**

1. *Geometry & the level set* — background mesh, `InterpolateToP1`, `CutInfo`, `dCut`.
2. *Stationary two-phase Stokes* — doubled spaces, Nitsche coupling, surface tension, ghost penalty.
3. *Time stepping* — transporting the level set and rebuilding the cut system.
4. *The rising bubble* — putting it together.
5. *Supplementary A* — the robust `ngsxditto` mean-curvature variant.
6. *Supplementary B* — a fine-mesh benchmark run & animation.

In [ ]:
from netgen.geom2d import SplineGeometry
from ngsolve import *
from xfem import *
import numpy as np

import os
try:
    if os.environ.get("NO_WEBGUI"):       # the website ships this notebook pre-rendered
        raise ImportError                 # (matplotlib only) — webgui needs a live kernel
    from ngsolve.webgui import Draw
    from xfem import DrawDC
    HAVE_WEBGUI = True
except Exception:
    HAVE_WEBGUI = False
    def Draw(*a, **k): pass
    def DrawDC(*a, **k): pass

### Physical parameters (Hysing–Turek, "case 1")

Index `0` is the **bubble** (the `{φ<0}` region), index `1` is the surrounding
**heavier fluid** (`{φ>0}`).  The bubble is lighter and less viscous, so
buoyancy makes it rise; surface tension keeps it compact.

In [ ]:
mu    = [1.0, 10.0]      # dynamic viscosity   [bubble, outside]
rho   = [100.0, 1000.0]  # density             [bubble, outside]
sigma = 24.5             # surface tension coefficient
g     = 0.98             # gravity
gvec  = CF((0, -g))      # gravity points downwards

R, cx, cy = 0.25, 0.5, 0.5    # initial bubble: circle of radius R centred at (cx,cy)

## 1. Geometry and the level set

In an **unfitted** (CutFEM) method the mesh does **not** resolve the interface.
We use a fixed background mesh of the whole channel `[0,1]×[0,2]` and describe
the bubble *implicitly* by the zero level set of a function

$$\varphi(x) < 0 \ \text{inside the bubble}, \qquad \varphi(x) = 0 \ \text{on the interface}, \qquad \varphi(x) > 0 \ \text{outside.}$$

We start from the signed distance of a circle.

In [ ]:
geo = SplineGeometry()
geo.AddRectangle((0, 0), (1, 2), bc="wall")     # bc="wall": one name for all four sides
mesh = Mesh(geo.GenerateMesh(maxh=0.06))
d = mesh.dim
print("background mesh:", mesh.ne, "elements")

### The P1 level set: `InterpolateToP1`

The cut machinery integrates over the part of each element where `φ<0`, `φ>0`,
or `φ=0`.  To make those cuts *exactly computable* it needs a level set that is
**piecewise linear (P1)** — then the zero set inside every triangle is a
straight segment.  `InterpolateToP1` takes an arbitrary expression and produces
that P1 representative.

In [ ]:
levelset = sqrt((x-cx)**2 + (y-cy)**2) - R     # exact (smooth) level set
lsetp1 = GridFunction(H1(mesh, order=1))       # the P1 representative used for cutting
InterpolateToP1(levelset, lsetp1)

if HAVE_WEBGUI:
    Draw(lsetp1, mesh, "lsetp1")

### Classifying elements: `CutInfo`

`CutInfo` inspects the P1 level set and labels every element.  The useful
*domain types* are

| type      | meaning                                            |
|-----------|----------------------------------------------------|
| `NEG`     | element lies entirely in `{φ<0}` (inside)          |
| `POS`     | element lies entirely in `{φ>0}` (outside)         |
| `IF`      | element is **cut** by the interface                |
| `HASNEG`  | element has *some* `{φ<0}` part (`NEG ∪ IF`)        |
| `HASPOS`  | element has *some* `{φ>0}` part (`POS ∪ IF`)        |

`GetElementsOfType` returns a `BitArray` marking the elements of a given type.

In [ ]:
ci = CutInfo(mesh, lsetp1)
for t, name in [(NEG,"NEG"),(POS,"POS"),(IF,"IF"),(HASNEG,"HASNEG"),(HASPOS,"HASPOS")]:
    print(f"{name:7s}: {ci.GetElementsOfType(t).NumSet():4d} elements")

### Cut integration: `dCut`

`dCut(lsetp1, domain_type)` is a *differential symbol* (like `dx`) that
integrates only over the requested part of the cut elements.  Let us check it
against the known geometry of a circle of radius `R=0.25`:

$$\text{area}=\pi R^2 \approx 0.1963, \qquad \text{perimeter}=2\pi R \approx 1.5708 .$$

In [ ]:
area = Integrate(CF(1) * dCut(lsetp1, NEG), mesh)   # area of {phi<0}
peri = Integrate(CF(1) * dCut(lsetp1, IF),  mesh)   # length of the interface {phi=0}
print(f"area      = {area:.5f}   (exact {np.pi*R**2:.5f})")
print(f"perimeter = {peri:.5f}   (exact {2*np.pi*R:.5f})")

The small mismatch is exactly the **geometry error** of the P1 level set: the
true circle is approximated by straight segments.  Using a higher-order
(isoparametric) level set would reduce it — but we deliberately keep the simple
P1 variant here.

We can also *see* the unfittedness: the interface (white) cuts straight through
the background mesh, ignoring element boundaries.

In [ ]:
if HAVE_WEBGUI:
    DrawDC(lsetp1, -1.0, 1.0, mesh, "inside(-1)/outside(+1)")   # two-valued field across the cut

## 2. Stationary two-phase Stokes

Now we solve, **for a fixed interface**, the two-phase Stokes problem: in each
subdomain $i\in\{0,1\}$

$$-\operatorname{div}\,\boldsymbol\sigma_i = \rho_i\,\mathbf g, \qquad
  \operatorname{div}\mathbf u_i = 0,\qquad
  \boldsymbol\sigma_i = 2\mu_i\,\boldsymbol\varepsilon(\mathbf u_i) - p_i\,\mathbb I ,$$

coupled across the interface by

$$[\![\mathbf u]\!]=0 \quad\text{(continuous velocity)}, \qquad
  [\![\boldsymbol\sigma\,\mathbf n]\!]=\sigma\,\kappa\,\mathbf n \quad\text{(surface tension)} .$$

### Doubled / restricted spaces (Hansbo's idea)

On a **cut** element the solution has *two* branches — one for each phase.  We
therefore take a standard Taylor–Hood pair and keep **two independent copies**,
one living on all `HASNEG` elements, one on all `HASPOS` elements.  `Compress`
throws away the unused dofs (the dofs of elements that do not carry that phase),
so each copy is as small as possible.  On cut elements *both* copies are active
— that is the Hansbo doubling.

In [ ]:
order = 2                                       # Taylor-Hood P2/P1
Vb = VectorH1(mesh, order=order, dirichlet="wall")
Qb = H1(mesh, order=order-1)

Vneg = Compress(Vb, GetDofsOfElements(Vb, ci.GetElementsOfType(HASNEG)))
Vpos = Compress(Vb, GetDofsOfElements(Vb, ci.GetElementsOfType(HASPOS)))
Qneg = Compress(Qb, GetDofsOfElements(Qb, ci.GetElementsOfType(HASNEG)))
Qpos = Compress(Qb, GetDofsOfElements(Qb, ci.GetElementsOfType(HASPOS)))

# one big product space: (vel_neg, vel_pos) x (p_neg, p_pos) x a single scalar
# (the scalar 'NumberSpace' is a Lagrange multiplier that pins the pressure level)
W = FESpace([Vneg*Vpos, Qneg*Qpos, NumberSpace(mesh)], dgjumps=True)
print("total dofs:", W.ndof)

gfup = GridFunction(W)
gfu, gfp, gfn = gfup.components       # gfu=(u_neg,u_pos), gfp=(p_neg,p_pos), gfn=number

`dgjumps=True` tells the space to reserve the extra couplings between neighbouring
elements that the ghost-penalty terms (below) will need.

### Integration regions and ghost-penalty facets

We need three integration symbols and, for the stabilization, the **ring band**
of facets between the cut elements and their same-phase neighbours.
`GetFacetsWithNeighborTypes` selects exactly those facets; `dFacetPatch`
integrates over the element patch glued along each such facet.

In [ ]:
dxs    = tuple(dCut(lsetp1, dt) for dt in [NEG, POS])   # bulk integrals, per phase
dGamma = dCut(lsetp1, IF)                               # interface integral
# (we call it dGamma, not ds, so the plain ngsolve boundary measure 'ds' stays
#  free for the DG transport in Part 3)

ba_facets = [GetFacetsWithNeighborTypes(mesh, a=ci.GetElementsOfType(HASNEG),
                                              b=ci.GetElementsOfType(IF)),
             GetFacetsWithNeighborTypes(mesh, a=ci.GetElementsOfType(HASPOS),
                                              b=ci.GetElementsOfType(IF))]
dw = tuple(dFacetPatch(definedonelements=f) for f in ba_facets)

### The weak form

We use a few helpers.  `kap` are the **Hansbo cut-ratio weights** (`CutRatioGF`
gives, per element, the volume fraction on the `NEG` side); they make the
averaged flux *stable independently of how the interface cuts the element*.

In [ ]:
h = specialcf.mesh_size
n_lset = 1.0/Norm(grad(lsetp1)) * grad(lsetp1)        # interface normal (from the level set)
kap = [CutRatioGF(ci), 1.0 - CutRatioGF(ci)]          # Hansbo weights (sum to 1)

lam   = 0.5*(mu[0]+mu[1]) * 20 * order * order        # Nitsche penalty
gp_v, gp_p = 0.1, 0.1                                  # ghost-penalty parameters

u, p, num = W.TrialFunction()
v, q, m   = W.TestFunction()

def eps(w):           return 0.5*(Grad(w) + Grad(w).trans)
def sig(i, w, pp):    return -2*mu[i]*eps(w[i]) + pp[i]*Id(d)        # Cauchy stress
def avg_flux(w, pp):  return sum(kap[i]*sig(i, w, pp)*n_lset for i in range(2))  # Hansbo average
def avg_grad(w):      return sum(kap[1-i]*Grad(w[i]) for i in range(2))          # averaged gradient
def jump(w):          return w[0] - w[1]

Now the bilinear / linear forms.  Reading them block by block:

* **per phase** — viscous term $2\mu_i\,\varepsilon(u):\varepsilon(v)$, the
  velocity/pressure (divergence) coupling, and the buoyancy load $\rho_i\,\mathbf g\!\cdot\! v$;
* **pressure level** — the `NumberSpace` multiplier fixes $\int p\,$ (otherwise
  pressure is only defined up to a constant);
* **Nitsche interface** — the consistent flux term plus its adjoint, and the
  penalty $\frac{\lambda}{h}[\![u]\!][\![v]\!]$ that weakly enforces $[\![u]\!]=0$;
* **surface tension** — the **Laplace–Beltrami** form
  $-\sigma\!\int_\Gamma (\mathbb I-\mathbf n\!\otimes\!\mathbf n):\nabla v_{\text{avg}}$.
  This is $\sigma\!\int_\Gamma \operatorname{div}_\Gamma v$, which is the surface
  tension force **without** ever computing the curvature explicitly;
* **ghost penalty** — couples each cut dof to its neighbours across the ring
  band; this restores stability/conditioning on arbitrarily small cuts (and
  inf-sup for the pressure).

In [ ]:
P = Id(d) - OuterProduct(n_lset, n_lset)     # tangential projection at the interface

a = BilinearForm(W, symmetric=False)
f = LinearForm(W)

for i in [0, 1]:                                            # per-phase bulk terms
    a += 2*mu[i]*InnerProduct(eps(u[i]), eps(v[i])) * dxs[i]
    a += (-div(u[i])*q[i] - div(v[i])*p[i]) * dxs[i]
    f += rho[i]*gvec*v[i] * dxs[i]                          # buoyancy load

a += (num*q[0] + m*p[0]) * dxs[0]                           # fix the pressure level

a += (avg_flux(u, p)*jump(v) + avg_flux(v, p)*jump(u)) * dGamma # Nitsche consistency + adjoint
a += lam/h * jump(u)*jump(v) * dGamma                           # Nitsche penalty

f += -sigma * InnerProduct(P, avg_grad(v)) * dGamma         # Laplace-Beltrami surface tension

for i in [0, 1]:                                            # ghost penalty (velocity & pressure)
    a += gp_v/h**2 * (u[i]-u[i].Other())*(v[i]-v[i].Other()) * dw[i]
    a += -gp_p     * (p[i]-p[i].Other())*(q[i]-q[i].Other()) * dw[i]

In [ ]:
with TaskManager():
    a.Assemble()
    f.Assemble()
    gfup.vec.data = a.mat.Inverse(W.FreeDofs(), inverse="umfpack") * f.vec
print("solved.  max |u| =", abs(gfup.components[0].vec.FV().NumPy()).max())

### Does it make sense?  The Laplace law

A clean check that the surface tension is implemented correctly is the **Laplace
law**: across a circular interface the pressure jump must be

$$p_{\text{in}} - p_{\text{out}} = \sigma\,\kappa = \frac{\sigma}{R} = \frac{24.5}{0.25} = 98 .$$

(We measure it *locally on the interface*; comparing volume-averaged pressures
would be polluted by the hydrostatic gradient from gravity.)

In [ ]:
per   = Integrate(CF(1)*dGamma, mesh)
pjump = Integrate((gfp.components[0]-gfp.components[1])*dGamma, mesh) / per
print(f"interface-averaged (p_in - p_out) = {pjump:6.2f}   (Laplace sigma/R = {sigma/R:.1f})")

if HAVE_WEBGUI:
    DrawDC(lsetp1, gfu.components[0], gfu.components[1], mesh, "velocity")
    DrawDC(lsetp1, gfp.components[0], gfp.components[1], mesh, "pressure")

## 3. Time stepping: moving the interface

So far the interface was frozen.  To let the bubble rise, the level set must be
**transported** by the computed velocity field $\mathbf u$:

$$\partial_t \varphi + \mathbf u\cdot\nabla\varphi = 0 .$$

We discretise this with the simplest robust explicit scheme — **upwind
Discontinuous Galerkin** on an `L2` field, advanced by explicit Euler.  Because
the velocity is (almost) divergence free we may use the conservative flux form.

In [ ]:
fes_phi = L2(mesh, order=2, dgjumps=True)
phi = GridFunction(fes_phi)
phi.Set(levelset)                            # initial level set in the transport space

gf_w = GridFunction(VectorH1(mesh, order=order))   # holds the advecting velocity each step

u_l, v_l = fes_phi.TnT()
n_F  = specialcf.normal(d)
flux = gf_w * n_F
upw  = IfPos(flux, u_l, u_l.Other())         # upwind value across a facet
conv = BilinearForm(fes_phi, nonassemble=True)       # 'nonassemble' -> we only use .Apply()
conv += -u_l * (gf_w * grad(v_l)) * dx
conv += flux * upw * (v_l - v_l.Other()) * dx(skeleton=True)   # interior facets
conv += IfPos(flux, flux*u_l, 0) * v_l * ds(skeleton=True)     # outflow boundary
invm = fes_phi.Mass(1).Inverse()             # L2 mass is block diagonal -> cheap inverse
res  = phi.vec.CreateVector()

def transport(dt_total, nsub):
    "Advance phi by dt_total using nsub explicit sub-steps, then refresh the P1 cut field."
    for _ in range(nsub):
        conv.Apply(phi.vec, res)
        phi.vec.data -= (dt_total/nsub) * invm * res
    lsetp1.Set(phi)        # project the (discontinuous) transported field to the P1 cut field
    ci.Update(lsetp1)      # re-classify elements for the new interface position

We sub-cycle the transport (`nsub` small steps per Stokes solve) to respect the
explicit CFL limit cheaply — the Stokes solve is the expensive part, the
transport is not.

### One reusable Stokes solve

When the interface moves, the set of active dofs changes, so the doubled spaces
and the whole system **must be rebuilt every step**.  We simply wrap *exactly
the assembly from Part 2* in a function.  (For a production code one would reuse
matrices with `RestrictedBilinearForm`; rebuilding is the clearest version.)

In [ ]:
def solve_stokes():
    Vneg = Compress(Vb, GetDofsOfElements(Vb, ci.GetElementsOfType(HASNEG)))
    Vpos = Compress(Vb, GetDofsOfElements(Vb, ci.GetElementsOfType(HASPOS)))
    Qneg = Compress(Qb, GetDofsOfElements(Qb, ci.GetElementsOfType(HASNEG)))
    Qpos = Compress(Qb, GetDofsOfElements(Qb, ci.GetElementsOfType(HASPOS)))
    W = FESpace([Vneg*Vpos, Qneg*Qpos, NumberSpace(mesh)], dgjumps=True)
    gfup = GridFunction(W)

    dxs    = tuple(dCut(lsetp1, dt) for dt in [NEG, POS])
    dGamma = dCut(lsetp1, IF)
    baf = [GetFacetsWithNeighborTypes(mesh, a=ci.GetElementsOfType(HASNEG), b=ci.GetElementsOfType(IF)),
           GetFacetsWithNeighborTypes(mesh, a=ci.GetElementsOfType(HASPOS), b=ci.GetElementsOfType(IF))]
    dw = tuple(dFacetPatch(definedonelements=fa) for fa in baf)
    n_lset = 1.0/Norm(grad(lsetp1)) * grad(lsetp1)
    kap = [CutRatioGF(ci), 1.0 - CutRatioGF(ci)]
    P = Id(d) - OuterProduct(n_lset, n_lset)

    u, p, num = W.TrialFunction(); v, q, m = W.TestFunction()
    def eps(w):          return 0.5*(Grad(w)+Grad(w).trans)
    def sig(i, w, pp):   return -2*mu[i]*eps(w[i]) + pp[i]*Id(d)
    def avg_flux(w, pp): return sum(kap[i]*sig(i, w, pp)*n_lset for i in range(2))
    def avg_grad(w):     return sum(kap[1-i]*Grad(w[i]) for i in range(2))
    def jump(w):         return w[0]-w[1]

    a = BilinearForm(W, symmetric=False); f = LinearForm(W)
    for i in [0, 1]:
        a += 2*mu[i]*InnerProduct(eps(u[i]), eps(v[i])) * dxs[i]
        a += (-div(u[i])*q[i] - div(v[i])*p[i]) * dxs[i]
        f += rho[i]*gvec*v[i] * dxs[i]
    a += (num*q[0] + m*p[0]) * dxs[0]
    a += (avg_flux(u, p)*jump(v) + avg_flux(v, p)*jump(u)) * dGamma
    a += lam/h * jump(u)*jump(v) * dGamma
    f += -sigma * InnerProduct(P, avg_grad(v)) * dGamma
    for i in [0, 1]:
        a += gp_v/h**2 * (u[i]-u[i].Other())*(v[i]-v[i].Other()) * dw[i]
        a += -gp_p     * (p[i]-p[i].Other())*(q[i]-q[i].Other()) * dw[i]
    a.Assemble(); f.Assemble()
    gfup.vec.data = a.mat.Inverse(W.FreeDofs(), inverse="umfpack") * f.vec
    return gfup

### The time loop

The algorithm of one step is:

1. **solve** the two-phase Stokes problem on the current geometry → velocity $\mathbf u$;
2. build the single advecting field $\mathbf u = \mathbf u_{\text{pos}}$ outside, $\mathbf u_{\text{neg}}$ inside;
3. **transport** the level set and **re-classify** the mesh;
4. repeat.

We monitor the bubble's centroid height, its rise velocity, and its area (mass
conservation) — a good unfitted scheme keeps the area nearly constant *without*
any reinitialisation over this time span.

In [ ]:
dt, tend, nsub = 0.005, 1.0, 5      # the simple level set stays clean to ~t=1; see note below
nsteps = int(tend/dt + 0.5)

A0 = Integrate(CF(1)*dCut(lsetp1, NEG), mesh)
hist = {"t": [0.0], "yc": [Integrate(y*dCut(lsetp1,NEG),mesh)/A0], "area": [A0]}
shapes = [(0.0, lsetp1.vec.FV().NumPy().copy())]
print(f"t=0.000  y_c={hist['yc'][0]:.4f}  area={A0:.5f}")

with TaskManager():
    for step in range(1, nsteps+1):
        gfup = solve_stokes()
        gu = gfup.components[0]
        gf_w.Set(IfPos(lsetp1, gu.components[1], gu.components[0]))
        transport(dt, nsub)

        t = step*dt
        A  = Integrate(CF(1)*dCut(lsetp1, NEG), mesh)
        yc = Integrate(y*dCut(lsetp1, NEG), mesh)/A
        hist["t"].append(t); hist["yc"].append(yc); hist["area"].append(A)
        if abs(t - round(t*4)/4) < dt/2:        # snapshot every 0.25 time units
            shapes.append((t, lsetp1.vec.FV().NumPy().copy()))
        if step % 20 == 0 or step == nsteps:
            vy = Integrate(gu.components[0][1]*dCut(lsetp1,NEG),mesh)/A
            print(f"t={t:5.3f}  y_c={yc:6.4f}  rise v_y={vy:6.3f}  "
                  f"area={A:.5f} ({100*(A-A0)/A0:+.2f}%)")
print("done")

## 4. The rising bubble

Finally we look at the result.  The left panel overlays the interface every
0.25 time units (it rises and flattens into a cap); the right panels show the
centroid height climbing roughly linearly and the bubble area staying nearly
constant.

In [ ]:
import matplotlib.pyplot as plt
verts = np.array([list(v.point) for v in mesh.vertices])

fig, ax = plt.subplots(1, 3, figsize=(13, 6), gridspec_kw={"width_ratios":[1,1,1]})
cols = plt.cm.viridis(np.linspace(0, 1, len(shapes)))
for (t_, lv), c in zip(shapes, cols):
    ax[0].tricontour(verts[:,0], verts[:,1], lv, levels=[0.0], colors=[c])
ax[0].set_xlim(0,1); ax[0].set_ylim(0,1.6); ax[0].set_aspect("equal")
ax[0].set_title("interface every 0.25 t-units\n(dark=early, bright=late)")

ax[1].plot(hist["t"], hist["yc"]); ax[1].set_xlabel("t"); ax[1].set_ylabel("centroid height")
ax[1].set_title("bubble rises"); ax[1].grid(alpha=.3)

ax[2].plot(hist["t"], 100*(np.array(hist["area"])-A0)/A0)
ax[2].set_xlabel("t"); ax[2].set_ylabel("area change [%]")
ax[2].set_title("mass conservation"); ax[2].grid(alpha=.3)
fig.tight_layout()

In [ ]:
# final shape, velocity field across the cut (inside + outside in one picture)
if HAVE_WEBGUI:
    gu = gfup.components[0]
    DrawDC(lsetp1, gu.components[0], gu.components[1], mesh, "final velocity")

## Summary

We built a complete two-phase rising-bubble solver with nothing but NGSolve and
ngsxfem, introducing each unfitted-FEM ingredient on the way:

* **`InterpolateToP1` + `CutInfo`** — implicit geometry and element marking;
* **`dCut` / `dFacetPatch`** — integration over cut regions and facet patches;
* **`Compress`-ed doubled spaces** — the Hansbo representation of a two-phase field;
* **Nitsche coupling** with **Hansbo (`CutRatioGF`) weights** — the interface conditions;
* **Laplace–Beltrami** surface tension — curvature-free surface tension;
* **ghost penalty** on the **ring band** — stability on small cuts and inf-sup;
* **explicit upwind-DG transport** + rebuild — the moving-interface time loop.

Every choice was the *simplest* one, and the limitations are exactly the ones
you would expect from that:

* the **P1 geometry** leaves a visible area error (~1 %) and produces *parasitic
  currents* at the interface (the surface-tension force and the discrete pressure
  gradient do not balance exactly);
* without **reinitialisation** the transported level set slowly drifts away from
  a signed-distance function, so beyond `t ≈ 1` the volume is no longer conserved
  and the interface starts to roughen — which is why we stop at `t = 1`.

Natural next steps for accuracy each slot directly into the framework above: a
higher-order **isoparametric** level set (smaller geometry error, far weaker
parasitic currents), a **space–time** discretisation of the transport, and
periodic **reinitialisation** of the level set for long-time runs.

The supplement below makes this concrete: it solves the *same* problem with the
high-level **ngsxditto** library — which bundles exactly those improvements
(isoparametric geometry, reinitialisation, a proper curvature solver) — and uses
it to run the full **Hysing–Turek benchmark** with a smooth animation.

---
## Supplementary A — a robust variant: `ngsxditto` with a mean-curvature solver

*(Supplementary — needs the **ngsxditto** add-on. Skip if you only want the from-scratch story.)*

Everything above we built *by hand* to expose the concepts.  In practice one uses
a library.  Here is the **identical** rising-bubble problem solved with
**ngsxditto**, whose high-level objects bundle the production-grade choices that
we deliberately skipped:

| our hand-built notebook            | ngsxditto                                        |
|------------------------------------|--------------------------------------------------|
| P1 level set (geometry error)      | **isoparametric** level set (`LevelSetGeometry`) |
| explicit DG transport, no reinit   | DG transport **+ FastMarching reinitialisation** |
| Laplace–Beltrami (parasitic curr.) | dedicated **`MeanCurvatureSolver`**              |
| rebuild the cut system every step  | incremental `TwoPhaseTaylorHood` + `TimeLoop`    |

We keep the **same discretisation order — Taylor–Hood P2/P1** — as the hand-built
version, so the comparison is fair; the difference is purely the production
machinery above.  Because the volume stays tight and the bubble stays smooth, this
variant runs the **Hysing–Turek benchmark to `t ≈ 2`** — capturing the two reference
extrema (max rise velocity at `t≈0.92`, min circularity at `t≈1.9`) — and compares
the quantities of interest with the reference.

> **Note**
>
> The two `ngsxditto` cells below are shown as the **production recipe** (ngsxditto is
> pinned to a development build the public site cannot reproduce, so they are not
> auto-executed). The **executed fine-mesh result** — the rising-bubble animation — is
> shown in **Supplement B** below.

In [ ]:
import math
from ngsxditto.utils.loglevel import loggingSlider
loggingSlider(default_level="WARNING"); loggingSlider("ngsxditto", default_level="WARNING")
from ngsxditto import *
from netgen.occ import *

# Keep the build machine's memory in check: ngsxditto factorises the two-phase
# system with Pardiso, whose footprint grows with the thread count — four threads
# is plenty here and keeps the peak well within a laptop's RAM.
SetNumThreads(4)

# ---- geometry (same channel, named edges) ----
maxh_x, dt_x, end_x = 0.06, 0.005, 2.0          # moderate mesh, fits in memory; t=2 captures both QoI extrema
ord_g, ord_v = 2, 2                              # geometry order 2 (isoparametric), velocity P2 (Taylor-Hood P2/P1)
domain = MoveTo(0, 0).Rectangle(1, 2).Face()
domain.edges.Min(X).name = "left";   domain.edges.Max(X).name = "right"
domain.edges.Min(Y).name = "bottom"; domain.edges.Max(Y).name = "top"
mesh_x = Mesh(OCCGeometry(domain, dim=2).GenerateMesh(maxh=maxh_x))

g_x = 0.98
rho1, mu1 = 1000.0, 10.0        # outside
rho2, mu2 = 100.0, 1.0          # bubble
sig_x = 24.5
print("ngsxditto mesh:", mesh_x.ne, "elements")

In [ ]:
# ---- level set with transport + reinitialisation (the two missing ingredients) ----
t_x = Parameter(0)
transport = ExplicitDGTransport(mesh_x, dt=dt_x, order=ord_g, compile=False)
levelset_x = LevelSetGeometry(transport, redistancing=FastMarching(),
                              autoredistancing=PeriodicRedistancing(40))
levelset_x.Initialize(sqrt((x-0.5)**2 + (y-0.5)**2) - 0.25)

# ---- two-phase Taylor-Hood P2/P1 fluid (isoparametric, proper curvature surface tension) ----
fluid1 = FluidParameters(viscosity=mu2, density=rho2, surface_tension_coeff=sig_x)  # bubble
fluid2 = FluidParameters(viscosity=mu1, density=rho1, surface_tension_coeff=sig_x)  # outside
mean_curv = MeanCurvatureSolver(mesh_x, order=ord_g, lset=levelset_x, gp_param=1); mean_curv.Step()
grav = CF((0, -g_x))
fluid = TwoPhaseTaylorHood(mesh_x, fluid1_params=fluid1, fluid2_params=fluid2,
                           lset=levelset_x, surface_tension=mean_curv.H, dt=dt_x,
                           order=ord_v, ghost_stab=1, nitsche_stab=100,        # order=2 -> P2/P1
                           f1=rho2*grav, f2=rho1*grav, add_convection=False, time_order=1)
fluid.SetOuterBoundaryCondition(StrongDirichletBC(region="top|bottom", values=CF((0, 0))))
fluid.SetOuterBoundaryCondition(NitscheNormalVelocityBC(region="left|right", values=CF(0)))
fluid.Initialize()

# the interface velocity is extended off the interface to drive the transport
vel_ext = LevelsetBasedExtension(levelset_x, gamma=1e-3, order=ord_g)
vel_ext.SetRhs(fluid.gfu.components[0]); levelset_x.transport.SetWind(vel_ext.field)

In [ ]:
# ---- diagnostics (benchmark QoI) + animation-frame collection ----
qoi = {"t": [], "yc": [], "vr": [], "circ": [], "area": []}
u_neg_y = fluid.gfu.components[0][1]
def diag():
    A = Integrate(CF(1)*levelset_x.dx_neg, mesh_x)
    if A <= 1e-12: return
    qoi["t"].append(t_x.Get()); qoi["area"].append(A)
    qoi["yc"].append(Integrate(y*levelset_x.dx_neg, mesh_x)/A)
    qoi["vr"].append(Integrate(u_neg_y*levelset_x.dx_neg, mesh_x)/A)
    per = Integrate(CF(1)*levelset_x.dS, mesh_x)
    qoi["circ"].append(2*math.sqrt(math.pi*A)/per if per > 0 else 0.0)

# velocity-quiver sampling grid (mesh points found ONCE -> fast)
vertsx = np.array([list(v.point) for v in mesh_x.vertices])
gx = np.linspace(0.06,0.94,12); gy = np.linspace(0.08,1.85,26)
GX,GY = np.meshgrid(gx,gy); ptsx = np.column_stack([GX.ravel(),GY.ravel()])
mpsx = [mesh_x(px,py) for px,py in ptsx]
framesx = []
uc_x = IfPos(levelset_x.lsetp1, fluid.gfu.components[1], fluid.gfu.components[0])
def grabx():
    U = np.array([uc_x(mp)[0] for mp in mpsx]); V = np.array([uc_x(mp)[1] for mp in mpsx])
    ins = np.array([levelset_x.lsetp1(mp) < 0.02 for mp in mpsx])
    framesx.append(dict(t=t_x.Get(), ls=levelset_x.lsetp1.vec.FV().NumPy().copy(),
                        U=U, V=V, ins=ins))

time_loop = TimeLoop(time=t_x, dt=dt_x, end_time=end_x,
                     display_progress_bar=False, show_profiles=False)
time_loop.Register(fluid,       name="moving stokes")
time_loop.Register(vel_ext,     name="vel ext.")
time_loop.Register(levelset_x,  name="levelset")
time_loop.Register(mean_curv,   name="mean curvature")
time_loop.Register(FunctionCallStepper(diag),  name="qoi",    step_frequency=4)
time_loop.Register(FunctionCallStepper(grabx), name="frames", step_frequency=12)

diag(); grabx()                      # t=0
try:
    time_loop()
    print("run completed")
except Exception as ex:               # late-time aggregation break -> keep what we have
    print(f"stopped early at t={t_x.Get():.3f}: {type(ex).__name__}")
print(f"ngsxditto: {len(framesx)} animation frames up to t={qoi['t'][-1]:.2f}, "
      f"area drift {100*(qoi['area'][-1]/qoi['area'][0]-1):+.2f}%")

## Supplementary B — a fine-mesh benchmark run & animation

*(Supplementary.)* With isoparametric geometry, reinitialisation and a proper curvature
force the bubble stays smooth and develops the clean ellipsoidal cap all the way through
the benchmark — no roughening, no volume loss. Here is the **pre-computed fine-mesh
`ngsxditto` run** to `t ≈ 2`, showing the interface and the interior velocity field as the
bubble rises and flattens into the characteristic cap:

![Fine-mesh ngsxditto rising bubble — interface and velocity field from t=0 to t≈2](https://raw.githubusercontent.com/schruste/ngsum2026-colab/colab/data/rising_bubble_ngsxditto.gif)

> **Note**
>
> The animation above is the **executed fine-mesh result**. The two cells below are the
> **recipe** that produces it — the `matplotlib` animation and the benchmark comparison
> against the Hysing–Turek reference values — shown but not auto-executed here (ngsxditto is
> pinned to a development build the public site cannot reproduce).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation
import matplotlib.tri as mtri
from IPython.display import HTML

def animate_frames(frames, verts, pts, ny=1.7):
    tri = mtri.Triangulation(verts[:,0], verts[:,1])
    fig, ax = plt.subplots(figsize=(3.2, 5.6))
    def draw(i):
        ax.clear(); fr = frames[i]
        ax.tricontourf(tri, np.where(fr["ls"]<0, -1.0, 1.0), levels=[-2,0,2],
                       colors=["#7fc7ff", "#eaf4fb"])
        ax.tricontour(tri, fr["ls"], levels=[0.0], colors="#08306b", linewidths=2)
        m = fr["ins"]
        ax.quiver(pts[m,0], pts[m,1], fr["U"][m], fr["V"][m], color="#08306b",
                  scale=8, width=0.006, alpha=0.7)
        ax.set_xlim(0,1); ax.set_ylim(0,ny); ax.set_aspect("equal")
        ax.set_title(f"t = {fr['t']:.2f}"); ax.set_xticks([]); ax.set_yticks([])
    anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=120)
    plt.close(fig)
    return HTML(anim.to_jshtml())

animate_frames(framesx, vertsx, ptsx)

And the **benchmark comparison**: the dashed lines are the Hysing–Turek reference
values (case 1) — max rise velocity ≈ 0.242 at `t≈0.92`, min circularity ≈ 0.901
at `t≈1.9`, centroid `y(3)≈1.081`.  The curves follow them closely, and the
bubble area stays essentially constant — exactly what the hand-built version
could *not* deliver past `t≈1`.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 3.8))
ax[0].plot(qoi["t"], qoi["yc"]); ax[0].axhline(1.081, ls="--", c="r")
ax[0].set_title("centroid height\n(ref y(3)=1.081)"); ax[0].set_xlabel("t")
ax[1].plot(qoi["t"], qoi["vr"]); ax[1].axhline(0.2417, ls="--", c="r")
ax[1].axvline(0.92, ls=":", c="r", alpha=.6)
ax[1].set_title("rise velocity\n(ref max 0.2417 @ t=0.92)"); ax[1].set_xlabel("t")
ax[2].plot(qoi["t"], qoi["circ"]); ax[2].axhline(0.9013, ls="--", c="r")
ax[2].axvline(1.9, ls=":", c="r", alpha=.6); ax[2].set_ylim(0.86, 1.01)
ax[2].set_title("circularity\n(ref min 0.9013 @ t=1.9)"); ax[2].set_xlabel("t")
ax[3].plot(qoi["t"], 100*(np.array(qoi["area"])/qoi["area"][0]-1))
ax[3].set_title("area drift [%]"); ax[3].set_xlabel("t")
for a_ in ax: a_.grid(alpha=.3)
fig.tight_layout()

This is where the tutorial ends and the **Grand Expedition** truly begins. Pack your coffee. ☕

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("14-melting-chocolate", "14 · Melting the chocolate 🍫☕")
    _next = None
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))